# Charging Flow Scenario

API-only sequence for a normal charging session on one station.

In [ ]:
import os
import time
import requests
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv('.env', usecwd=True)
load_dotenv(env_path if env_path else None)

API_BASE_URL = os.getenv('VCP_API_BASE_URL', 'https://vcp.pionix.com').rstrip('/')
STATION_ID = os.getenv('VCP_STATION_ID')
EVSE_ID = os.getenv('VCP_EVSE_ID', '1')
CONNECTOR_ID = os.getenv('VCP_CONNECTOR_ID', '1')
AUTH_ID_TOKEN = os.getenv('VCP_AUTH_ID_TOKEN', 'DEADBEEF')
AUTH_ID_TOKEN_TYPE = os.getenv('VCP_AUTH_ID_TOKEN_TYPE', 'ISO14443')
CHARGE_WAIT_SECONDS = int(os.getenv('VCP_CHARGE_WAIT_SECONDS', '10'))

if not STATION_ID:
    raise ValueError('VCP_STATION_ID is missing in .env')

BASE = f"{API_BASE_URL}/{STATION_ID}/api"
CP = f"{BASE}/cp"
EV = f"{BASE}/ev"

def post(url, payload=None, params=None):
    response = requests.post(url, json=payload, params=params, timeout=30)
    qp = f" params={params}" if params else ""
    print(f"POST {url}{qp} -> {response.status_code}")
    try:
        print(response.json())
    except ValueError:
        print(response.text)
    response.raise_for_status()

print(f"Loaded env from: {env_path}")
print(f"Station base: {BASE}")

In [ ]:
# Start session
cp_params = {"evse_id": EVSE_ID, "connector_id": CONNECTOR_ID}

# CP endpoints expect query params (snake_case), not JSON body fields.
post(f"{CP}/plugin", params={**cp_params, "halfway": "true"})
post(f"{EV}/plugin", {"evseId": EVSE_ID, "connectorId": CONNECTOR_ID, "soc": 25})
time.sleep(2)
post(
    f"{CP}/authorize",
    params={
        **cp_params,
        "id": AUTH_ID_TOKEN,
        "type": AUTH_ID_TOKEN_TYPE,
    },
)
time.sleep(2)
post(f"{EV}/state", params={"ready": "true"})

In [ ]:
# Charge, then stop
cp_params = {"evse_id": EVSE_ID, "connector_id": CONNECTOR_ID}
print(f"Waiting {CHARGE_WAIT_SECONDS}s while charging...")
time.sleep(CHARGE_WAIT_SECONDS)
post(f"{EV}/plugout", {"evseId": EVSE_ID, "connectorId": CONNECTOR_ID})
time.sleep(1)
post(f"{CP}/plugout", params=cp_params)
time.sleep(2)
post(f"{EV}/end", {})